In [1]:
!nvidia-smi

Tue Nov 17 19:59:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(props.total_memory / 1024**3, 2),
        "GB"
    )

CUDA available: True
GPU: NVIDIA L4
VRAM: 22.03 GB


In [3]:
%pip install "transformers[torch,vision,audio] @ git+https://github.com/swiss-ai/transformers.git@apertusv1p5-release-pre-upstream"

  Cloning https://github.com/swiss-ai/transformers.git (to revision apertusv1p5-release-pre-upstream) to /tmp/pip-install-8oxi4v75/transformers_edc9723a0b9f40f4823e5c23d96a288c
  Running command git clone --filter=blob:none --quiet https://github.com/swiss-ai/transformers.git /tmp/pip-install-8oxi4v75/transformers_edc9723a0b9f40f4823e5c23d96a288c
  Running command git checkout -q 3797303dda74844e3d1f8977ff5518bb91f818b4
  Resolved https://github.com/swiss-ai/transformers.git to commit 3797303dda74844e3d1f8977ff5518bb91f818b4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 141.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.9 MB/s  0:00:00
  

In [1]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/Crosslingual-Behavioral-Testing-of-Semantic-Robustness-in-NLP-Models

/content/drive/MyDrive/Crosslingual-Behavioral-Testing-of-Semantic-Robustness-in-NLP-Models


# Load Model

In [4]:
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor
)

MODEL_ID = "swiss-ai/Apertus-v1.5-8B"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
).eval()

print("Apertus loaded successfully!")

processor_config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.61k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.03k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 44.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

[transformers] CUDA-fused xIELU not available (No module named 'xielu') – falling back to a Python version.
For CUDA xIELU (experimental), `pip install git+https://github.com/nickjbrowning/XIELU`


Loading weights:   0%|          | 0/924 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Apertus loaded successfully!


# Zero-shot Prompting

In [5]:
SYSTEM_PROMPT = """
You are an expert Natural Language Inference (NLI) classifier.

Determine the logical relationship between a premise
and a hypothesis.

There are exactly three possible labels:

- Entailment: If the premise is true, the hypothesis
  must also be true.
- Contradiction: The premise and hypothesis cannot
  both be true.
- Neutral: The premise neither entails nor contradicts
  the hypothesis.

Base your decision only on the provided information.
Do not introduce unsupported assumptions.

Respond with exactly ONE lowercase label:
entailment, neutral, or contradiction.

Do not provide explanations or additional text.
"""


def predict_nli(premise, hypothesis):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Premise: {premise}\n"
                f"Hypothesis: {hypothesis}\n"
                "NLI label:"
            )
        }
    ]

    # Format the conversation using Apertus's chat template
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate the answer
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False
        )

    # Decode only the newly generated tokens
    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    response = processor.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return response

In [8]:
# Test examples before processing the dataset
test_examples = [
    {
        "premise": "The student did not pass the examination.",
        "hypothesis": "The student passed the examination.",
        "gold": "contradiction"
    },
    {
        "premise": "All three students passed the examination.",
        "hypothesis": "At least one of the three students passed.",
        "gold": "entailment"
    },
    {
        "premise": "Some students passed the examination.",
        "hypothesis": "All students passed the examination.",
        "gold": "neutral"
    }
]

for example in test_examples:

    prediction = predict_nli(
        example["premise"],
        example["hypothesis"]
    )

    print("Premise:", example["premise"])
    print("Hypothesis:", example["hypothesis"])
    print("Gold:", example["gold"])
    print("Prediction:", prediction)
    print("-" * 50)

Premise: The student did not pass the examination.
Hypothesis: The student passed the examination.
Gold: contradiction
Prediction: contradiction
--------------------------------------------------
Premise: All three students passed the examination.
Hypothesis: At least one of the three students passed.
Gold: entailment
Prediction: entailment
--------------------------------------------------
Premise: Some students passed the examination.
Hypothesis: All students passed the examination.
Gold: neutral
Prediction: contradiction
--------------------------------------------------


#

In [9]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Crosslingual-Behavioral-Testing-of-Semantic-Robustness-in-NLP-Models/english_checklist_tests.csv")

print(df.head())
print("Total examples:", len(df))

          case_id    test_id language phenomenon test_type  pair_role  \
0  en_neg_01_orig  en_neg_01       en   negation       MFT   original   
1  en_neg_01_pert  en_neg_01       en   negation       MFT  perturbed   
2  en_neg_02_orig  en_neg_02       en   negation       MFT   original   
3  en_neg_02_pert  en_neg_02       en   negation       MFT  perturbed   
4  en_neg_03_orig  en_neg_03       en   negation       DIR   original   

                                             premise  \
0  The safety inspector rejected the damaged bridge.   
1  The safety inspector rejected the damaged bridge.   
2  The defendant was not dishonest during the trial.   
3  The defendant was not dishonest during the trial.   
4  All passengers boarded the morning train on time.   

                                      hypothesis     gold_label  \
0      The inspector did not approve the bridge.     entailment   
1       The inspector did not reject the bridge.  contradiction   
2   The defendant was t

In [10]:
df = df[
    df["validated"].astype(str).str.lower() == "true"
].copy()

df = df[df["language"] == "en"].copy()

In [11]:
predictions = []

for index, row in df.iterrows():

    prediction = predict_nli(
        row["premise"],
        row["hypothesis"]
    )

    predictions.append(prediction)

    if len(predictions) % 10 == 0:
        print(f"Processed {len(predictions)}/{len(df)}")

Processed 10/30
Processed 20/30
Processed 30/30


In [12]:
df["apertus_raw_prediction"] = predictions

In [18]:
import re, os

VALID_LABELS = {
    "entailment",
    "neutral",
    "contradiction"
}

def parse_label(response):

    first_line = response.strip().splitlines()[0]
    cleaned = first_line.strip().lower().strip(
        " *`#:.\"'"
    )

    match = re.fullmatch(
        r"(?:label\s*:\s*)?(entailment|neutral|contradiction)",
        cleaned
    )

    if match:
        return match.group(1)

    return "invalid"


df["prediction"] = df[
    "apertus_raw_prediction"
].apply(parse_label)

df["correct"] = (
    df["prediction"] == df["gold_label"].str.lower()
)

df.to_csv(
    "apertus_english_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

In [14]:
#
# OUTPUT_DIR = (
#     "/content/drive/MyDrive/Crosslingual-Behavioral-Testing-of-Semantic-Robustness-in-NLP-Models"
# )

# os.makedirs(OUTPUT_DIR, exist_ok=True)

In [15]:
# df.to_csv(
#     f"{OUTPUT_DIR}/apertus_predictions.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# Evaluation

In [16]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

valid = df[df["prediction"] != "invalid"].copy()

print(
    "Accuracy:",
    accuracy_score(
        valid["gold_label"],
        valid["prediction"]
    )
)

print(
    "Macro F1:",
    f1_score(
        valid["gold_label"],
        valid["prediction"],
        labels=[
            "entailment",
            "neutral",
            "contradiction"
        ],
        average="macro"
    )
)

print(
    classification_report(
        valid["gold_label"],
        valid["prediction"],
        labels=[
            "entailment",
            "neutral",
            "contradiction"
        ]
    )
)

Accuracy: 0.8333333333333334
Macro F1: 0.5994397759103641
               precision    recall  f1-score   support

   entailment       1.00      0.89      0.94        18
      neutral       0.00      0.00      0.00         1
contradiction       0.90      0.82      0.86        11

     accuracy                           0.83        30
    macro avg       0.63      0.57      0.60        30
 weighted avg       0.93      0.83      0.88        30

